In [1]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import mode

In [3]:
import sys
sys.path.append("/Users/mariahloehr/IICD/IICD/feature_importance")

In [4]:
import locomp
from locomp import *
from locomp.MLmodels import *
from locomp.util_locomp import *
import itertools
import importlib
from sklearn.base import BaseEstimator, RegressorMixin, clone
import itertools
from functools import partial
import multiprocessing as mp
import re

import functions_case as il
import importlib

In [9]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/T47D.csv")

X = df.drop(columns=['Metadata_well', 'phase'])
y= df['Metadata_well']

feature_names = X.columns.tolist()
X = X.to_numpy()
y = y.to_numpy()

X_train, X_test, y_train, y_tes = train_test_split(X, y, test_size=0.2, random_state=949, stratify=y)

In [16]:
# Define RBF-kernel SVM
def svm(X,Y):
    fit = SVC(kernel='rbf', C = 400, gamma = 0.01, probability=True, random_state=949
              ).fit(X,Y)
    return fit

In [17]:
J1 = 0
J2 = 1
m_ratio = 0.5
n_ratio = 0.5
B = 5000
fit_func = svm

In [18]:
predictions, in_mp_obs, in_mp_feature = predictMPClass(X_train, y_train, X_test, n_ratio, m_ratio, B, fit_func, n_jobs = -1)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
/Users/mariahloehr/IICD/IICD/feature_importance/locomp/util_locomp.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  idx_I =Y_pd.groupby(0, group_keys=False).apply(lambda x: x.sample(frac=n_ratio))
/Users/mariahloehr/IICD/IICD/feature_importance/locomp/util_locomp.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  idx_I =Y_pd.groupby(0, 

In [21]:
# 0.5 Step 1: convert one-hot predictions to class indices

from sklearn.metrics import accuracy_score, cohen_kappa_score

class_preds = predictions.argmax(axis=2)   # shape (B, N)

# Step 2: majority vote per sample
majority_vote = mode(class_preds, axis=0, keepdims=False).mode  # shape (N,)

# Define the mapping from class index → treatment label
class_labels = np.array([0, 1, 10, 100, 1000])

# Convert indices to labels
mode_pred = class_labels[majority_vote]

acc = accuracy_score(y_tes, mode_pred)
kappa = cohen_kappa_score(y_tes, mode_pred)

print(f"Minipatch model accuracy: {acc:.3f}")
print(f"Minipatch model Cohen's kappa: {kappa:.3f}")

Minipatch model accuracy: 0.715
Minipatch model Cohen's kappa: 0.639


In [22]:
# === Load existing results DataFrame ===
results_df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/LOCO/cancer_mp_results.csv", index_col=0)

# === Set values ===
model_name = "SVM MP"  # or whatever is appropriate
results_df.loc[model_name, 'Accuracy'] = acc

# === Save updated results ===
results_df.to_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/LOCO/cancer_mp_results.csv")

In [23]:
# save cohen results
# === Load existing results ===
results_df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/cancer_cohen_results.csv", index_col=0)

# === Insert values ===
model_name = "SVM MP"
results_df.loc[model_name, "Cohen's Kappa"] = kappa

# === Save updated file ===
results_df.to_csv("/Users/mariahloehr/IICD/IICD/Cancer treatment/cancer_cohen_results.csv")